# CheXpert architecture comparison

This notebook runs the image-only pipeline on Google Colab: data preparation, SimCLR pretraining, supervised and SimCLR downstream comparisons, aggregation, and optional locked final evaluation. Keep the repository, dataset, and generated artifacts on Google Drive so they persist between sessions. Start with the smoke profile.

## 1. Connect Drive and set paths

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

# Change these paths to match your Drive.
PROJECT_ROOT = Path('/content/drive/MyDrive/cv-contrastive-learning')
DATASET_ROOT = Path('/content/drive/MyDrive/CheXpert-v1.0-small')
PROFILE = 'smoke'  # 'full' runs all five backbones, budgets, seeds, and epochs

assert (PROJECT_ROOT / 'pyproject.toml').is_file(), f'Repository not found: {PROJECT_ROOT}'
assert (DATASET_ROOT / 'train.csv').is_file(), f'train.csv not found: {DATASET_ROOT}'
assert (DATASET_ROOT / 'valid.csv').is_file(), f'valid.csv not found: {DATASET_ROOT}'

## 2. Install and check the runtime

In [ ]:
%cd $PROJECT_ROOT
%pip install -e .
import json, torch, yaml, subprocess, sys
print('Python:', sys.version.split()[0], 'PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Select a GPU runtime in Runtime > Change runtime type before training.')
if PROFILE == 'full' and not torch.cuda.is_available():
    raise RuntimeError('The full experiment matrix requires a GPU runtime.')

## 3. Prepare manifests and patient cohorts

The official validation manifest is prepared for held-out evaluation only. Do not use it to choose architectures, prompts, or settings.

In [ ]:
if PROFILE == 'smoke':
    architectures, budgets, seeds = ['resnet18'], [1.0], [42]
    pretrain_epochs, downstream_epochs, batch_size, image_size = 1, 1, 8, 224
else:
    architectures = yaml.safe_load((PROJECT_ROOT / 'configs/models/backbones.yaml').read_text())['architectures']
    budgets, seeds = [0.01, 0.05, 0.10, 0.25, 1.0], [42, 43, 44]
    pretrain_epochs, downstream_epochs, batch_size, image_size = 200, 50, 8, 224

subprocess.run([sys.executable, 'scripts/prepare_data.py', '--dataset-root', str(DATASET_ROOT), '--frontal-only-downstream'], cwd=PROJECT_ROOT, check=True)
manifest_dir = PROJECT_ROOT / 'data/processed/manifests'
if PROFILE == 'smoke':
    import pandas as pd
    smoke_dir = PROJECT_ROOT / 'data/processed/colab_smoke'
    smoke_dir.mkdir(parents=True, exist_ok=True)
    pretrain_frame = pd.read_csv(manifest_dir / 'pretrain_train.csv').head(128)
    train_frame = pd.read_csv(manifest_dir / 'downstream_train.csv')
    development_frame = pd.read_csv(manifest_dir / 'development.csv').head(256)
    smoke_patients = sorted(train_frame['patient_id'].astype(str).unique())[:64]
    train_frame = train_frame[train_frame['patient_id'].astype(str).isin(smoke_patients)]
    pretrain_frame.to_csv(smoke_dir / 'pretrain.csv', index=False)
    train_frame.to_csv(smoke_dir / 'train.csv', index=False)
    development_frame.to_csv(smoke_dir / 'development.csv', index=False)
    (smoke_dir / 'patients.json').write_text(json.dumps({'seed': 42, 'label_fraction': 1.0, 'patient_ids': smoke_patients}, indent=2))
print('Architectures:', architectures, 'Budgets:', budgets, 'Seeds:', seeds)

## 4. Run the training matrix

The smoke profile checks one short ResNet-18 run through all three methods. The full profile runs all five backbones across three methods, five patient-level budgets, and three seeds; it requires substantial GPU time. Run outputs and resolved configs are stored on Drive.

In [ ]:
def write_config(config, name):
    path = PROJECT_ROOT / 'outputs/notebook_configs' / f'{name}.yaml'
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(yaml.safe_dump(config, sort_keys=False))
    return path

def run_script(script, config_path):
    subprocess.run([sys.executable, str(PROJECT_ROOT / script), '--config', str(config_path)], cwd=PROJECT_ROOT, check=True)

def run_or_reuse(script, config, name, run_dir):
    config_path = write_config(config, name)
    metrics_path = run_dir / 'metrics.json'
    metadata_path = run_dir / 'run_metadata.json'
    if metrics_path.is_file() and metadata_path.is_file():
        saved_config = json.loads(metadata_path.read_text())['config']
        if saved_config != config:
            raise ValueError(f'Completed run config differs from requested config: {run_dir}')
        return
    run_script(script, config_path)

base_pretrain = yaml.safe_load((PROJECT_ROOT / 'configs/pretrain/simclr.yaml').read_text())
base_downstream = {
    'supervised': yaml.safe_load((PROJECT_ROOT / 'configs/downstream/supervised.yaml').read_text()),
    'simclr_linear': yaml.safe_load((PROJECT_ROOT / 'configs/downstream/simclr_linear.yaml').read_text()),
    'simclr_finetune': yaml.safe_load((PROJECT_ROOT / 'configs/downstream/simclr_finetune.yaml').read_text()),
}

for architecture in architectures:
    pretrain = dict(base_pretrain, encoder=architecture, image_size=image_size, batch_size=batch_size, num_workers=2, epochs=pretrain_epochs)
    pretrain['manifest'] = str(smoke_dir / 'pretrain.csv' if PROFILE == 'smoke' else manifest_dir / 'pretrain_train.csv')
    pretrain_dir = PROJECT_ROOT / 'outputs/pretrain' / f'simclr-{architecture}-{PROFILE}-seed42'
    pretrain['output_dir'] = str(pretrain_dir)
    run_or_reuse('scripts/train_pretrain.py', pretrain, f'pretrain-{architecture}-{PROFILE}', pretrain_dir)

    for seed in seeds:
        for budget in budgets:
            budget_label = f'{budget:g}'.replace('.', 'p')
            cohort = smoke_dir / 'patients.json' if PROFILE == 'smoke' else PROJECT_ROOT / 'data/processed/splits' / f'budget_{budget_label}_seed_{seed}.json'
            for method, template in base_downstream.items():
                config = dict(template, encoder=architecture, seed=seed, image_size=image_size, batch_size=batch_size, num_workers=2, epochs=downstream_epochs, label_fraction=budget, sampled_patients=str(cohort))
                config['train_manifest'] = str(smoke_dir / 'train.csv' if PROFILE == 'smoke' else manifest_dir / 'downstream_train.csv')
                config['development_manifest'] = str(smoke_dir / 'development.csv' if PROFILE == 'smoke' else manifest_dir / 'development.csv')
                run_dir = PROJECT_ROOT / 'outputs/downstream' / f'{method}-{architecture}-{budget_label}-{PROFILE}-seed{seed}'
                config['output_dir'] = str(run_dir)
                if method != 'supervised':
                    config['simclr_checkpoint'] = str(pretrain_dir / 'best.pt')
                run_or_reuse('scripts/train_downstream.py', config, f'{method}-{architecture}-{budget_label}-{PROFILE}-seed{seed}', run_dir)
print('Training matrix finished.')

## 5. Aggregate development metrics

Results are grouped by architecture, method, and label budget from saved metric artifacts.

In [ ]:
subprocess.run([sys.executable, 'scripts/aggregate_results.py', '--root', str(PROJECT_ROOT / 'outputs'), '--output-dir', str(PROJECT_ROOT / 'outputs/reports')], cwd=PROJECT_ROOT, check=True)
import pandas as pd
results_path = PROJECT_ROOT / 'outputs/reports/label_efficiency_summary.csv'
if results_path.is_file():
    display(pd.read_csv(results_path).sort_values(['architecture', 'method', 'label_fraction']))

## 6. Locked final evaluation

Use this section only after choosing a model using development results. Set `LOCKED_RUN_DIR` to the selected run directory; the official validation set is evaluated once for that locked configuration.

In [ ]:
LOCKED_RUN_DIR = None  # Set to a selected run folder under outputs/downstream.
if LOCKED_RUN_DIR is not None:
    locked_dir = Path(LOCKED_RUN_DIR)
    checkpoint = torch.load(locked_dir / 'best.pt', map_location='cpu', weights_only=False)
    locked_config = checkpoint['config']
    evaluation = yaml.safe_load((PROJECT_ROOT / 'configs/evaluation/final_validation.yaml').read_text())
    evaluation.update(
        encoder=locked_config['encoder'],
        image_size=locked_config['image_size'],
        checkpoint=str(locked_dir / 'best.pt'),
        thresholds=str(locked_dir / 'thresholds.json'),
        manifest=str(manifest_dir / 'final_validation.csv'),
        output_dir=str(PROJECT_ROOT / 'outputs/reports' / f'final-validation-{locked_dir.name}'),
        seed=int(locked_config['seed']),
    )
    run_script('scripts/evaluate_checkpoint.py', write_config(evaluation, f'final-validation-{locked_dir.name}'))
else:
    print('Final validation skipped. Set LOCKED_RUN_DIR after development-based selection.')